<a href="https://colab.research.google.com/github/Ganasa18/belajar-tensorflow/blob/main/train_class_prompt_labeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q datasets pandas requests tqdm scikit-learn sentence-transformers joblib

In [2]:
from datasets import load_dataset

ds = load_dataset(
    "ilhamfadheel/alpaca-cleaned-indonesian",
    split="train"
)

print(ds)
print(ds[0])

README.md:   0%|          | 0.00/10.0k [00:00<?, ?B/s]

AlpacaCleaned_translated_id.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

AlpacaCleaned_translated_id.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/45631 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 45631
})
{'instruction': 'Sebutkan tiga senyawa organik.', 'input': '', 'output': 'Tiga senyawa organik adalah glukosa (C6H12O6), metana (CH4), dan etanol (C2H5OH).'}


In [3]:
# IMPORT DATA SET
SAMPLE_SIZE = 5000

ds_small = ds.shuffle(seed=42).select(
    range(min(SAMPLE_SIZE, len(ds)))
)

def build_prompt(row):
    instruction = row["instruction"].strip()
    input_text = row["input"].strip()

    if input_text:
        return instruction + "\n" + input_text

    return instruction

prompts = [build_prompt(x) for x in ds_small]

print("Jumlah prompt:", len(prompts))

for x in prompts[:5]:
    print("\n---")
    print(x)

Jumlah prompt: 5000

---
Beri saya pertanyaan untuk ditanyakan kepada seseorang yang Anda kenal.

---
Jelaskan faktor-faktor yang berkontribusi terhadap krisis ekonomi global saat ini.

---
Jelaskan seperti apa hari-hari musim panas pada umumnya di gurun.

---
Tulis ulang kalimat berikut untuk mendapatkan arti yang berbeda: "Saya memakan apelnya."

---
Tuliskan sebuah cerita tentang seorang wanita yang bertahan melewati kesulitan.


In [4]:
# SIMPLE
# Fakta sederhana, lookup, klasifikasi sederhana,
# jawaban singkat, operasi mudah.

# GENERAL
# Penjelasan umum yang tidak membutuhkan reasoning berat.

# REASONING
# Analisis, matematika, planning, comparison,
# trade-off, deduksi, multi-step problem.

# CODING_SIMPLE
# Generate snippet sederhana, regex, SQL sederhana,
# fungsi kecil, syntax.

# CODING_COMPLEX
# Debugging kompleks, architecture, multi-file,
# race condition, performance, security review.

# TRANSFORM
# Translation, summarization, rewrite,
# extract, shorten, formatting.

# CREATIVE
# Story, brainstorming kreatif, slogan,
# dialog, ide kreatif.

LABELS = [
    "SIMPLE",
    "GENERAL",
    "REASONING",
    "CODING_SIMPLE",
    "CODING_COMPLEX",
    "TRANSFORM",
    "CREATIVE"
]

In [5]:
import os

BASE_URL = "https://openrouter.ai/api/v1/chat/completions"
API_KEY = "ISI_API_KEY"
MODEL = "openrouter/free"

TEMPERATURE = 0
TIMEOUT = 90

In [7]:
from google.colab import userdata
API_KEY = userdata.get('OPEN_ROUTER')

In [8]:
SYSTEM_PROMPT = """
You are labeling user prompts for an AI model router.

Classify each prompt into EXACTLY ONE label:

SIMPLE
- Simple factual questions
- Easy lookup
- Basic classification
- Very short/simple task
- Does not require substantial reasoning

GENERAL
- General explanation
- Normal knowledge question
- Moderate general assistant task
- Does not clearly belong to another category

REASONING
- Mathematics
- Multi-step reasoning
- Planning
- Comparison and trade-offs
- Analysis
- Deduction
- Complex decision making

CODING_SIMPLE
- Small code snippets
- Simple functions
- Regex
- Basic SQL
- Syntax questions
- Straightforward programming tasks

CODING_COMPLEX
- Complex debugging
- Software architecture
- Multi-component systems
- Race conditions
- Security/code review
- Performance optimization
- Complex implementation

TRANSFORM
- Translation
- Summarization
- Rewrite
- Shortening
- Extraction
- Reformatting
- Changing an existing text without requiring substantial new reasoning

CREATIVE
- Story writing
- Creative brainstorming
- Dialogue
- Slogans
- Fiction
- Creative ideation

Important rules:

1. Classify based on USER INTENT, not individual keywords.
2. The prompt may be Indonesian, English, or mixed Indonesian-English.
3. Technical words do NOT automatically mean CODING.
4. Mentioning a language such as "Bahasa Inggris" does NOT automatically mean TRANSFORM.
5. "Explain" does NOT automatically mean REASONING.
6. Choose CODING_COMPLEX only if strong technical reasoning or debugging is actually required.
7. Use SIMPLE for genuinely easy tasks.
8. Return valid JSON only.

Output schema:

{
  "label": "ONE_LABEL",
  "difficulty": 1,
  "confidence": 0.95
}

difficulty must be integer 1-5.
confidence must be number 0-1.
"""

In [13]:
import requests
import time

def call_teacher(prompt):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": MODEL,
        "temperature": 0,
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    }

    print("  -> POST", BASE_URL)
    print("  -> model:", MODEL)

    start = time.time()

    response = requests.post(
        BASE_URL,
        headers=headers,
        json=payload,

        # jangan 90 detik dulu untuk debugging
        timeout=(10, 30)
    )

    print(
        f"  <- HTTP {response.status_code}"
        f" ({time.time() - start:.2f}s)"
    )

    response.raise_for_status()

    data = response.json()

    return data["choices"][0]["message"]["content"]

In [22]:
# PARSER JSON

import re

def parse_teacher_output(text):
    text = text.strip()

    match = re.search(r'\{[\s\S]*?\}', text)

    if not match:
        raise ValueError(
            f"Tidak menemukan JSON: {text[:300]}"
        )

    data = json.loads(match.group())

    label = data.get("label")
    difficulty = int(data.get("difficulty"))
    confidence = float(data.get("confidence"))

    if label not in LABELS:
        raise ValueError(f"Label invalid: {label}")

    if not 1 <= difficulty <= 5:
        raise ValueError("Difficulty invalid")

    if not 0 <= confidence <= 1:
        raise ValueError("Confidence invalid")

    return {
        "label": label,
        "difficulty": difficulty,
        "confidence": confidence
    }

In [15]:
test_prompt = """
review architecture backend saya dan cari kemungkinan
race condition serta bottleneck performance
"""

raw = call_teacher(test_prompt)

print("RAW:")
print(raw)

parsed = parse_teacher_output(raw)

print("\nPARSED:")
print(parsed)

  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (2.65s)
RAW:
{
  "label": "CODING_COMPLEX",
  "difficulty": 4,
  "confidence": 0.92
}

PARSED:
{'label': 'CODING_COMPLEX', 'difficulty': 4, 'confidence': 0.92}


In [23]:
import pandas as pd
import os

CHECKPOINT_PATH = "/content/router_teacher_checkpoint.csv"

if os.path.exists(CHECKPOINT_PATH):
    df_result = pd.read_csv(CHECKPOINT_PATH)

    print(
        "Resume checkpoint:",
        len(df_result),
        "prompt"
    )

else:
    df_result = pd.DataFrame(
        columns=[
            "prompt",
            "label",
            "difficulty",
            "confidence",
            "status"
        ]
    )

    print("Checkpoint belum ada, mulai dari 0")

print(df_result.head())
print("Rows:", len(df_result))

Resume checkpoint: 40 prompt
                                              prompt      label  difficulty  \
0  Beri saya pertanyaan untuk ditanyakan kepada s...     SIMPLE           1   
1  Jelaskan faktor-faktor yang berkontribusi terh...    GENERAL           3   
2  Jelaskan seperti apa hari-hari musim panas pad...     SIMPLE           1   
3  Tulis ulang kalimat berikut untuk mendapatkan ...  TRANSFORM           1   
4  Tuliskan sebuah cerita tentang seorang wanita ...   CREATIVE           3   

   confidence status  
0        0.95     ok  
1        0.93     ok  
2        0.95     ok  
3        0.95     ok  
4        0.95     ok  
Rows: 40


In [26]:
# LOOP LABELING

from tqdm.auto import tqdm
import time

START_INDEX = len(df_result)

SAVE_EVERY = 20
MAX_RETRIES = 3

results = df_result.to_dict("records")

print("Mulai dari index:", START_INDEX)
print("Total prompt:", len(prompts))

for i in tqdm(
    range(START_INDEX, len(prompts)),
    initial=START_INDEX,
    total=len(prompts)
):

    prompt = prompts[i]
    success = False

    for attempt in range(MAX_RETRIES):
        try:
            raw = call_teacher(prompt)
            parsed = parse_teacher_output(raw)

            results.append({
                "prompt": prompt,
                "label": parsed["label"],
                "difficulty": parsed["difficulty"],
                "confidence": parsed["confidence"],
                "status": "ok"
            })

            success = True
            break

        except Exception as e:
            print(
                f"\nError index {i}, "
                f"attempt {attempt + 1}/{MAX_RETRIES}: "
                f"{type(e).__name__}: {e}"
            )

            try:
                print("RAW RESPONSE:", raw[:500])
            except:
                pass

        time.sleep(2 * (attempt + 1))

    if not success:
        results.append({
            "prompt": prompt,
            "label": None,
            "difficulty": None,
            "confidence": None,
            "status": "error"
        })

    if (
        (i + 1) % SAVE_EVERY == 0
        or i == len(prompts) - 1
    ):
        pd.DataFrame(results).to_csv(
            CHECKPOINT_PATH,
            index=False
        )

        print(
            f"\nCheckpoint saved: "
            f"{len(results)}/{len(prompts)}"
        )

print("Labeling selesai")

Mulai dari index: 40
Total prompt: 5000


  1%|          | 40/5000 [00:00<?, ?it/s]

  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (1.56s)
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (3.33s)
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (1.46s)

Error index 42, attempt 1/3: ValueError: Tidak menemukan JSON: User Safety: safe
RAW RESPONSE: User Safety: safe
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (1.52s)
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (7.87s)
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (4.32s)
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (6.02s)
  -> POST https://openrouter.ai/api/v1/chat/completions
  -> model: openrouter/free
  <- HTTP 200 (17.77s)
  -> POST https://openrouter.ai/api/v

KeyboardInterrupt: 

In [27]:
# LOAD CHECKPOINT / FINAL RESULT

import pandas as pd

df = pd.read_csv(CHECKPOINT_PATH)

print("Total tersimpan:", len(df))
print()

print("Status:")
print(df["status"].value_counts(dropna=False))
print()

print("Label:")
print(df["label"].value_counts(dropna=False))
print()

print("Shape:", df.shape)

Total tersimpan: 100

Status:
status
ok    100
Name: count, dtype: int64

Label:
label
SIMPLE           35
CREATIVE         23
GENERAL          22
TRANSFORM        13
REASONING         5
CODING_SIMPLE     2
Name: count, dtype: int64

Shape: (100, 5)


In [29]:
# Hapus Request Gagal

df_clean = df[
    df["status"] == "ok"
].copy()

df_clean = df[
    df["status"] == "ok"
].copy()

df_clean.to_csv(
    "/content/router_seed_indonesia.csv",
    index=False
)

print("Seed Indonesia:", len(df_clean))

Seed Indonesia: 100


In [ ]:
# Distribusi label

print(
    df_clean["label"].value_counts()
)

print()

print(
    df_clean["label"]
    .value_counts(normalize=True)
    .round(3)
)

In [ ]:
# Filter Confidence

MIN_CONFIDENCE = 0.80

df_high = df_clean[
    df_clean["confidence"] >= MIN_CONFIDENCE
].copy()

print(
    "High confidence:",
    len(df_high)
)

print()

print(
    df_high["label"].value_counts()
)

In [ ]:
# Lihat Prompt tidak yakin

uncertain = df_clean.sort_values(
    "confidence"
)

print(
    uncertain[
        [
            "prompt",
            "label",
            "difficulty",
            "confidence"
        ]
    ]
    .head(30)
    .to_string(index=False)
)

In [ ]:
# Example Category

for label in LABELS:

    print("\n")
    print("=" * 70)
    print(label)
    print("=" * 70)

    samples = (
        df_high[
            df_high["label"] == label
        ]
        .sample(
            min(
                10,
                len(
                    df_high[
                        df_high["label"]
                        == label
                    ]
                )
            ),
            random_state=42
        )
    )

    for _, row in samples.iterrows():

        print(
            f"\n[{row['confidence']:.2f}]",
            row["prompt"][:300]
        )

In [ ]:
# Export Techer Data
FINAL_DATASET = (
    "/content/router_teacher_labeled.csv"
)

df_high.to_csv(
    FINAL_DATASET,
    index=False
)

print(FINAL_DATASET)

In [ ]:
# Training Clasifier

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

X = embedding_model.encode(
    df_high["prompt"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

y = df_high["label"].values

print(X.shape)

In [ ]:
# Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(
    len(X_train),
    len(X_test)
)

In [ ]:
# Train Logistic Regression
from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression(
    max_iter=2000,
    random_state=42
)

classifier.fit(
    X_train,
    y_train
)

print("Done")

In [ ]:
# Evaluate

from sklearn.metrics import (
    accuracy_score,
    classification_report
)

pred = classifier.predict(
    X_test
)

print(
    "Accuracy:",
    accuracy_score(
        y_test,
        pred
    )
)

print()

print(
    classification_report(
        y_test,
        pred,
        digits=3
    )
)

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix

labels_sorted = list(
    classifier.classes_
)

cm = confusion_matrix(
    y_test,
    pred,
    labels=labels_sorted
)

cm_df = pd.DataFrame(
    cm,
    index=labels_sorted,
    columns=labels_sorted
)

print(cm_df)

In [ ]:
# Test Prompt
test_prompts = [
    "berapa 15 persen dari 500",

    "jelaskan fungsi docker compose",

    "bandingkan REST dan GraphQL "
    "untuk backend aplikasi besar",

    "buat regex untuk validasi email",

    "debug kenapa service systemd saya "
    "restart terus dan cek kemungkinan "
    "race condition",

    "ringkas tulisan ini menjadi 5 poin",

    "terjemahkan kalimat ini ke bahasa jepang",

    "buat cerita pendek tentang robot kecil",

    "kenapa request ke API direct cepat "
    "tapi lewat router jadi lambat",

    "review architecture backend saya, "
    "analisis bottleneck, retry strategy, "
    "dan desain ulang flow request"
]

emb = embedding_model.encode(
    test_prompts,
    normalize_embeddings=True
)

prediction = classifier.predict(
    emb
)

probability = classifier.predict_proba(
    emb
)

for prompt, label, probs in zip(
    test_prompts,
    prediction,
    probability
):

    confidence = probs.max()

    print("\nPROMPT:")
    print(prompt)

    print(
        "ROUTE:",
        label
    )

    print(
        "CONFIDENCE:",
        round(
            float(confidence),
            3
        )
    )

In [ ]:
import joblib

MODEL_PATH = (
    "/content/router_classifier_v2.joblib"
)

joblib.dump(
    classifier,
    MODEL_PATH
)

print(MODEL_PATH)